## Summary: The Complete RAG Pipeline

### What We Built:

```
1. Data Ingestion
   ├── Text Files (TextLoader)
   ├── Web Pages (WebBaseLoader + BeautifulSoup)
   └── PDF Files (PyPDFLoader)
   
2. Text Processing
   └── Chunking (RecursiveCharacterTextSplitter)
       - Split into manageable pieces
       - Maintain context with overlap
   
3. Embedding Generation
   └── Convert text → vectors (OpenAIEmbeddings)
   
4. Vector Storage & Retrieval
   ├── Chroma (Easy, development-friendly)
   └── FAISS (Fast, production-ready)
   
5. Semantic Search
   └── Query → Find relevant chunks → Return to LLM
```

### Key Takeaways:

✅ **Multiple data sources**: Text, Web, PDF  
✅ **Chunking is essential**: Makes documents searchable and LLM-compatible  
✅ **Embeddings capture meaning**: Similar content = similar vectors  
✅ **Vector databases enable fast search**: Find relevant info in milliseconds  
✅ **Choice of vector store**: Chroma for simplicity, FAISS for performance  

### Next Steps:

This retrieval system can now be connected to an LLM to create a complete RAG application where:
- User asks a question
- System retrieves relevant chunks
- LLM generates answer based on retrieved context

In [ ]:
from langchain_community.vectorstores import FAISS

# Create a FAISS vector store with only the first 15 documents
# This is useful for:
# - Testing with a smaller dataset
# - Faster experimentation
# - Demonstrating the concept without processing all documents
db = FAISS.from_documents(documents[:15], OpenAIEmbeddings())

# Now you can use db.similarity_search(query) just like with Chroma
# Example:
# results = db.similarity_search("What is the transformer architecture?")
# print(results[0].page_content)

## Step 7: FAISS Vector Database (Alternative)

**What we're doing:**
- Using **FAISS** instead of Chroma as the vector store
- Only storing the first 15 document chunks (for demonstration)

**What is FAISS?**
- **Facebook AI Similarity Search** - developed by Meta
- Extremely fast and efficient vector search library
- Better for large-scale production systems (millions of vectors)
- Can run on CPU or GPU

**Chroma vs FAISS:**

| Feature | Chroma | FAISS |
|---------|--------|-------|
| Ease of Use | Simpler, beginner-friendly | More complex, powerful |
| Speed | Good for small/medium datasets | Optimized for large scale |
| Persistence | Built-in database | In-memory (needs separate storage) |
| Best For | Development, small apps | Production, high performance |

**Why use FAISS:**
- Production-grade performance
- Handles millions of embeddings efficiently
- Industry standard for vector search
- More control over indexing algorithms

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Create embeddings and store in Chroma vector database
# This does two things:
# 1. Converts all document chunks to vectors using OpenAI's embedding model
# 2. Stores them in Chroma for fast retrieval
db = Chroma.from_documents(documents, OpenAIEmbeddings())

# Define a search query
query = "Who are the authors of attention is all you need?"

# Perform similarity search - finds most relevant chunks
retrieved_results = db.similarity_search(query)

# Display the most relevant chunk's content
print(retrieved_results[0].page_content)

## Step 6: Vector Embeddings and Chroma Vector Store

**What we're doing:**
- Converting text chunks into **embeddings** (numerical vectors)
- Storing these embeddings in **Chroma** (a vector database)
- Performing **semantic search** to find relevant chunks

**Key Concepts:**

### Embeddings
- Text is converted to vectors (arrays of numbers) like `[0.12, -0.33, 0.85, ...]`
- Similar text = similar vectors (semantic similarity)
- OpenAI's embedding model understands meaning, not just keywords

### Vector Database (Chroma)
- Stores embeddings with metadata
- Performs fast similarity search using vector math
- Finds the "closest" chunks to your query in semantic space

### Similarity Search
- Your query gets embedded: `"Who are the authors?"` → vector
- Database finds chunks with closest vector distance
- Returns the most semantically relevant content

**Why:** This is the core of RAG - retrieve relevant context before generation

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Create a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Maximum characters per chunk
    chunk_overlap=200      # Overlap between chunks to preserve context
)

# Split the PDF documents into smaller chunks
documents = text_splitter.split_documents(docs)

# Display the first 5 chunks to see the result
documents[:5]

## Step 5: Text Splitting (Chunking)

**What we're doing:**
- Using `RecursiveCharacterTextSplitter` to break large documents into smaller chunks
- `chunk_size=1000`: Each chunk contains ~1000 characters
- `chunk_overlap=200`: Adjacent chunks share 200 characters to maintain context continuity

**Why this is critical:**
- LLMs have token limits - we can't send entire books to them
- Smaller chunks improve retrieval precision (find exact relevant sections)
- Overlap ensures important information at chunk boundaries isn't lost
- Better semantic search - smaller chunks = more focused content

**How it works:**
- Tries to split on paragraphs first, then sentences, then words
- "Recursive" means it intelligently preserves natural document structure

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Create a PDF loader for the attention paper
loader = PyPDFLoader('attention.pdf')

# Load all pages from the PDF
docs = loader.load()

# Display the loaded documents
docs

## Step 4: Data Ingestion from PDF Files

**What we're doing:**
- Using `PyPDFLoader` to extract text from PDF documents
- PDF is a complex format, and this loader handles the parsing automatically
- Each page becomes a separate document with metadata

**Why:** 
- PDFs are extremely common for research papers, reports, manuals, books
- Essential for building RAG systems that answer questions from PDF documents
- The "Attention is All You Need" paper is a famous AI research paper about Transformers

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

# Create a WebBaseLoader with specific HTML parsing rules
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            # Only extract content from these CSS classes
            class_=("post-title", "post-content", "post-header")
        )
    )
)

# Load and parse the webpage content
text_documents = loader.load()

# Display the loaded documents
text_documents

## Step 3: Data Ingestion from Web Pages

**What we're doing:**
- Using `WebBaseLoader` to scrape content from websites
- Using BeautifulSoup4 (bs4) to parse HTML and extract specific content
- `SoupStrainer` filters only the relevant parts of the webpage (post-title, post-content, post-header)

**Why:** 
- Web scraping allows us to build RAG systems from online documentation, blogs, articles
- Filtering specific CSS classes ensures we only get relevant content, not navigation/ads
- This is crucial for building knowledge bases from web sources

In [ ]:
from langchain_community.document_loaders import TextLoader

# Create a TextLoader instance pointing to the text file
loader = TextLoader("speech.txt")

# Load the document - returns a list of Document objects
text_documents = loader.load()

# Display the loaded documents
text_documents

## Step 2: Data Ingestion from Text Files

**What we're doing:**
- Using `TextLoader` to load plain text files
- This is the simplest form of document loading
- Perfect for .txt files containing speeches, articles, or any text content

**Why:** Text files are a common data source, and we need to load them into LangChain's document format for processing.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Set OpenAI API key from environment
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

## Step 1: Environment Setup

First, we need to load environment variables to access our API keys securely.

# Data Ingestion and Retrieval with LangChain

This notebook demonstrates the complete RAG (Retrieval-Augmented Generation) pipeline:
1. **Data Ingestion** - Loading documents from various sources
2. **Text Splitting** - Breaking documents into manageable chunks
3. **Vector Embeddings** - Converting text to numerical vectors
4. **Vector Stores** - Storing and searching embeddings efficiently